In [1]:
pip install anthropic pillow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install pandas openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import base64
import json
import re
import pandas as pd
import os
from dotenv import load_dotenv
from anthropic import Anthropic

# ۱. تنظیمات اولیه
load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def extract_table_data(image_path, image_media_type="image/gif"):
    image_data = encode_image(image_path)
    
    # پرومپت تخصصی شما (بدون تغییر در محتوا، فقط تاکید بر ترتیب JSON)
    system_instruction = """You are an elite Data Auditor specialized in historical hydrological yearbooks. 
    Your mission is to transcribe the daily discharge table with 100% precision.

    ### 1. SEMANTIC IDENTIFICATION:
    - IDENTIFY THE DAY COLUMN: Labels like 'Giorno', 'G.', or unlabeled numbers 1-31.
    - IDENTIFY MONTH COLUMNS: 12 columns (Gennaio to Dicembre).
    - TABLE BOUNDARIES: Focus ONLY on the 31x12 grid. Ignore summary rows (Media, Totale).

    ### 2. MANDATORY TWO-PASS AUDIT:
    *PASS 1: VERTICAL INTEGRITY* - Count numeric entries per month and match calendar days.
    *PASS 2: HORIZONTAL ALIGNMENT* - Ensure Day 15 aligns with row 15.

    ### 3. OUTPUT RULES:
    - Return a JSON list of 31 objects.
    - Each object MUST use these EXACT keys in order: "Giorno", "Gen", "Feb", "Mar", "Apr", "Mag", "Giu", "Lug", "Ago", "Set", "Ott", "Nov", "Dic".
    - Even if a month is empty, the key must exist with 'null'.
    - Return ONLY the raw JSON array. No conversational text."""

    

    user_instruction = "Extract the daily discharge data from this image for all months and all 31 days. Ensure each row corresponds to the 'GIORNO' column."

    message = client.messages.create(
        model="claude-sonnet-4-6", 
        max_tokens=4000, 
        temperature=0,
        system=system_instruction,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": image_media_type, "data": image_data}},
                    {"type": "text", "text": user_instruction}
                ],
            }
        ],
    )
    return message.content[0].text

# --- شروع فرآیند پردازش دسته‌جمعی ---

input_folder = "../test_pictures/" # پوشه ورودی عکس‌ها
columns_order = ["Giorno", "Gen", "Feb", "Mar", "Apr", "Mag", "Giu", "Lug", "Ago", "Set", "Ott", "Nov", "Dic"]

if not os.path.exists(input_folder):
    print(f"Error: Folder {input_folder} not found.")
else:
    files = [f for f in os.listdir(input_folder) if f.lower().endswith('.gif')]
    print(f"Found {len(files)} files. Starting extraction...")

    for filename in files:
        image_path = os.path.join(input_folder, filename)
        base_name = os.path.splitext(filename)[0]
        output_filename = f"{base_name}_claude.xlsx"

        print(f"Processing: {filename}...")
        
        try:
            raw_output = extract_table_data(image_path)
            
            # پیدا کردن JSON در خروجی
            clean_output = re.search(r'\[.*\]', raw_output, re.DOTALL)
            if clean_output:
                json_data = json.loads(clean_output.group(0))
                df = pd.DataFrame(json_data)
                
                # --- بخش اصلاحی برای جلوگیری از جابه‌جایی ستون‌ها ---
                # مطمئن می‌شویم تمام ستون‌های ماه‌ها وجود دارند
                for col in columns_order:
                    if col not in df.columns:
                        df[col] = None
                
                # مرتب‌سازی ستون‌ها دقیقاً بر اساس ترتیب تقویم
                df = df[columns_order]
                # ------------------------------------------------
                
                df.to_excel(output_filename, index=False)
                print(f"✅ Saved: {output_filename}")
            else:
                print(f"❌ No valid data for {filename}")
                
        except Exception as e:
            print(f"❌ Error in {filename}: {e}")

print("\nAll files processed successfully.")

Found 15 files. Starting extraction...
Processing: Q_Cagliari_1924_66.gif...
✅ Saved: Q_Cagliari_1924_66_claude.xlsx
Processing: Q_Bari_1969_26.gif...
✅ Saved: Q_Bari_1969_26_claude.xlsx
Processing: Q_Cagliari_1933_91.gif...
✅ Saved: Q_Cagliari_1933_91_claude.xlsx
Processing: Q_Roma_1980_30.gif...
✅ Saved: Q_Roma_1980_30_claude.xlsx
Processing: Q_Pescara_1947_73.gif...
✅ Saved: Q_Pescara_1947_73_claude.xlsx
Processing: Q_Genova_1936_86.gif...
✅ Saved: Q_Genova_1936_86_claude.xlsx
Processing: Q_Pescara_1926_65.gif...
✅ Saved: Q_Pescara_1926_65_claude.xlsx
Processing: Q_Venezia_1926_138.gif...
✅ Saved: Q_Venezia_1926_138_claude.xlsx
Processing: Q_Roma_1928_73.gif...
✅ Saved: Q_Roma_1928_73_claude.xlsx
Processing: Q_Venezia_1935_186.gif...
✅ Saved: Q_Venezia_1935_186_claude.xlsx
Processing: Q_Parma_1942_75.gif...
✅ Saved: Q_Parma_1942_75_claude.xlsx
Processing: Q_Pisa_1935_66.gif...
✅ Saved: Q_Pisa_1935_66_claude.xlsx
Processing: Q_Catanzaro_1937_36.gif...
✅ Saved: Q_Catanzaro_1937_36_cla

In [4]:
import pandas as pd
import os
import re

# ۱. تنظیمات پوشه‌ها
input_folder = "./" # پوشه‌ای که فایل‌های _claude.xlsx در آن هستند
output_folder = "./reformatted_results/" # پوشه برای ذخیره فایل‌های نهایی

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# ۲. نقشه تبدیل نام ماه‌ها به عدد
month_map = {
    'Gen': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'Mag': 5, 'Giu': 6,
    'Lug': 7, 'Ago': 8, 'Set': 9, 'Ott': 10, 'Nov': 11, 'Dic': 12
}

# ۳. پیدا کردن تمام فایل‌هایی که از API گرفتیم
files = [f for f in os.listdir(input_folder) if f.endswith('_claude.xlsx')]

print(f"Found {len(files)} files to reformat.")

for filename in files:
    file_path = os.path.join(input_folder, filename)
    
    try:
        # بارگذاری فایل کلاود
        df_claude = pd.read_excel(file_path)
        
        # استخراج سال از نام فایل (مثلاً از Q_Bari_1969_26 استخراج می‌کند 1969)
        year_match = re.search(r'\d{4}', filename)
        current_year = int(year_match.group(0)) if year_match else 0

        # عملیات تبدیل (Wide to Long)
        # فرض بر این است که ستون اول فایل‌های شما 'Giorno' نام دارد
        df_melted = df_claude.melt(id_vars=['Giorno'], var_name='Month_Name', value_name='Value')

        # ساخت ستون‌های نهایی طبق فرمت دستی (Manual)
        df_final = pd.DataFrame()
        df_final[0] = [current_year] * len(df_melted) # ستون اول: سال
        df_final[1] = df_melted['Month_Name'].map(month_map) # ستون دوم: ماه
        df_final[2] = df_melted['Giorno'] # ستون سوم: روز
        df_final[3] = pd.to_numeric(df_melted['Value'], errors='coerce').fillna(0) # ستون چهارم: مقدار

        # مرتب‌سازی بر اساس ماه و روز
        df_final = df_final.sort_values(by=[1, 2])

        # ساخت نام فایل خروجی (اسم اصلی + _reform)
        base_name = filename.replace('_claude.xlsx', '')
        output_name = f"{base_name}_reform.xlsx"
        
        # ذخیره بدون هدر و ایندکس (دقیقاً مثل فرمت دستی شما)
        df_final.to_excel(os.path.join(output_folder, output_name), index=False, header=False)
        
        print(f"✅ Reformat Done: {output_name}")

    except Exception as e:
        print(f"❌ Error in {filename}: {e}")

print("\nAll 15 files have been reformatted successfully!")

Found 15 files to reformat.
✅ Reformat Done: Q_Catanzaro_1937_36_reform.xlsx
✅ Reformat Done: Q_Roma_1927_79_reform.xlsx
✅ Reformat Done: Q_Parma_1942_75_reform.xlsx
✅ Reformat Done: Q_Roma_1980_30_reform.xlsx
✅ Reformat Done: Q_Pisa_1935_66_reform.xlsx
✅ Reformat Done: Q_Roma_1928_73_reform.xlsx
✅ Reformat Done: Q_Venezia_1935_186_reform.xlsx
✅ Reformat Done: Q_Cagliari_1933_91_reform.xlsx
✅ Reformat Done: Q_Cagliari_1924_66_reform.xlsx
✅ Reformat Done: Q_Bari_1969_26_reform.xlsx
✅ Reformat Done: Q_Catanzaro_1940_54_reform.xlsx
✅ Reformat Done: Q_Genova_1936_86_reform.xlsx
✅ Reformat Done: Q_Pescara_1947_73_reform.xlsx
✅ Reformat Done: Q_Pescara_1926_65_reform.xlsx
✅ Reformat Done: Q_Venezia_1926_138_reform.xlsx

All 15 files have been reformatted successfully!


In [5]:
import pandas as pd
import numpy as np
import os
import re

# ۱. تنظیمات پوشه‌ها
reformatted_folder = "./reformatted_results/" 
manual_folder = "../test_pictures/"          

reform_files = [f for f in os.listdir(reformatted_folder) if f.endswith('_reform.xlsx')]
all_accuracies = []

print(f"--- شروع مقایسه با اصلاح منطق NaN/Zero برای {len(reform_files)} فایل ---\n")

for ref_file in reform_files:
    base_name = ref_file.replace('_reform.xlsx', '')
    manual_file = f"{base_name}.xlsx"
    
    ref_path = os.path.join(reformatted_folder, ref_file)
    man_path = os.path.join(manual_folder, manual_file)
    
    if not os.path.exists(man_path):
        continue

    try:
        # ۲. بارگذاری فایل‌ها
        df_manual = pd.read_excel(man_path, header=None)
        df_reform = pd.read_excel(ref_path, header=None)

        # ۳. انتخاب ستون‌های ۱ تا ۳ (ماه، روز، مقدار)
        df_man_3cols = df_manual.iloc[:, 1:4].copy()
        df_ref_3cols = df_reform.iloc[:, 1:4].copy()

        df_man_3cols.columns = ['Month', 'Day', 'Value_Man']
        df_ref_3cols.columns = ['Month', 'Day', 'Value_Ref']

        # ۴. یکسان‌سازی مقادیر (تبدیل هر نوع NaN یا خالی به 0)
        # این خط باعث می‌شود هم 0 و هم خالی، یکسان دیده شوند
        df_man_3cols['Value_Man'] = pd.to_numeric(df_man_3cols['Value_Man'], errors='coerce').fillna(0)
        df_ref_3cols['Value_Ref'] = pd.to_numeric(df_ref_3cols['Value_Ref'], errors='coerce').fillna(0)

        # ۵. فیلتر کردن سال (بر اساس اسم فایل)
        year_match = re.search(r'\d{4}', base_name)
        if year_match:
            target_year = int(year_match.group(0))
            df_man_final = df_man_3cols[df_manual.iloc[:, 0] == target_year].copy()
        else:
            df_man_final = df_man_3cols.copy()

        # ۶. ادغام (Merge)
        comparison = pd.merge(df_man_final, df_ref_3cols, on=['Month', 'Day'])

        # ۷. محاسبه مطابقت (Accuracy)
        # حالا که همه NaNها 0 شدند، مقایسه عددی مستقیم انجام می‌دهیم
        comparison['Is_Match'] = np.isclose(
            comparison['Value_Man'].astype(float), 
            comparison['Value_Ref'].astype(float), 
            atol=1e-3
        )

        total_points = len(comparison)
        matches = comparison['Is_Match'].sum()
        accuracy = (matches / total_points) * 100 if total_points > 0 else 0

        print(f"✅ {base_name}: Accuracy = {accuracy:.2f}%")
        
        all_accuracies.append({
            'Table': base_name,
            'Total_Days': total_points,
            'Matches': matches,
            'Accuracy': accuracy
        })

    except Exception as e:
        print(f"❌ Error in {base_name}: {e}")

# ۸. خروجی نهایی
summary_df = pd.DataFrame(all_accuracies)
summary_df.to_excel("final_accuracy_batch_report_v2.xlsx", index=False)
print("\nگزارش نهایی در فایل 'final_accuracy_batch_report_v2.xlsx' ذخیره شد.")

--- شروع مقایسه با اصلاح منطق NaN/Zero برای 15 فایل ---

✅ Q_Cagliari_1933_91: Accuracy = 94.79%
✅ Q_Venezia_1935_186: Accuracy = 92.33%
✅ Q_Cagliari_1924_66: Accuracy = 91.80%
✅ Q_Catanzaro_1940_54: Accuracy = 80.33%
✅ Q_Bari_1969_26: Accuracy = 100.00%
✅ Q_Pescara_1926_65: Accuracy = 98.36%
✅ Q_Pescara_1947_73: Accuracy = 96.16%
✅ Q_Genova_1936_86: Accuracy = 81.15%
✅ Q_Venezia_1926_138: Accuracy = 96.44%
✅ Q_Parma_1942_75: Accuracy = 92.05%
✅ Q_Catanzaro_1937_36: Accuracy = 83.29%
✅ Q_Roma_1927_79: Accuracy = 92.60%
✅ Q_Roma_1980_30: Accuracy = 98.63%
✅ Q_Pisa_1935_66: Accuracy = 85.75%
✅ Q_Roma_1928_73: Accuracy = 95.36%

گزارش نهایی در فایل 'final_accuracy_batch_report_v2.xlsx' ذخیره شد.


In [6]:
import pandas as pd
import numpy as np
import os
import re

# ۱. تنظیمات پوشه‌ها
reformatted_folder = "./reformatted_results/" 
manual_folder = "../test_pictures/"          
debug_folder = "./debug_mismatches/" # پوشه جدید برای بررسی خطاها

if not os.path.exists(debug_folder):
    os.makedirs(debug_folder)

reform_files = [f for f in os.listdir(reformatted_folder) if f.endswith('_reform.xlsx')]
all_accuracies = []

print(f"--- شروع مقایسه و عیب‌یابی برای {len(reform_files)} فایل ---\n")

for ref_file in reform_files:
    base_name = ref_file.replace('_reform.xlsx', '')
    manual_file = f"{base_name}.xlsx"
    
    ref_path = os.path.join(reformatted_folder, ref_file)
    man_path = os.path.join(manual_folder, manual_file)
    
    if not os.path.exists(man_path):
        continue

    try:
        df_manual = pd.read_excel(man_path, header=None)
        df_reform = pd.read_excel(ref_path, header=None)

        # انتخاب ستون‌های اصلی
        df_man_3cols = df_manual.iloc[:, 1:4].copy()
        df_ref_3cols = df_reform.iloc[:, 1:4].copy()
        df_man_3cols.columns = ['Month', 'Day', 'Value_Man']
        df_ref_3cols.columns = ['Month', 'Day', 'Value_Ref']

        # یکسان‌سازی (تبدیل NaN و خالی به 0)
        df_man_3cols['Value_Man'] = pd.to_numeric(df_man_3cols['Value_Man'], errors='coerce').fillna(0)
        df_ref_3cols['Value_Ref'] = pd.to_numeric(df_ref_3cols['Value_Ref'], errors='coerce').fillna(0)

        # فیلتر سال
        year_match = re.search(r'\d{4}', base_name)
        target_year = int(year_match.group(0)) if year_match else None
        if target_year:
            df_man_final = df_man_3cols[df_manual.iloc[:, 0] == target_year].copy()
        else:
            df_man_final = df_man_3cols.copy()

        # ادغام
        comparison = pd.merge(df_man_final, df_ref_3cols, on=['Month', 'Day'])

        # محاسبه مطابقت
        comparison['Is_Match'] = np.isclose(
            comparison['Value_Man'].astype(float), 
            comparison['Value_Ref'].astype(float), 
            atol=1e-3
        )

        total_points = len(comparison)
        matches = comparison['Is_Match'].sum()
        accuracy = (matches / total_points) * 100 if total_points > 0 else 0

        # --- بخش عیب‌یابی (Debug) ---
        # اگر دقت از حد انتظار کمتر بود، مغایرت‌ها رو ذخیره کن
        if accuracy < 98.0:
            mismatches = comparison[comparison['Is_Match'] == False].copy()
            if not mismatches.empty:
                # اضافه کردن ستون تفاوت برای دیدن شدت خطا
                mismatches['Diff'] = mismatches['Value_Man'] - mismatches['Value_Ref']
                error_file = os.path.join(debug_folder, f"mismatches_{base_name}.xlsx")
                mismatches.to_excel(error_file, index=False)
                print(f"⚠️ {base_name}: Accuracy {accuracy:.2f}% (Saved {len(mismatches)} errors to debug folder)")
            else:
                print(f"✅ {base_name}: Accuracy {accuracy:.2f}%")
        else:
            print(f"✅ {base_name}: Accuracy {accuracy:.2f}%")
        
        all_accuracies.append({'Table': base_name, 'Accuracy': accuracy})

    except Exception as e:
        print(f"❌ Error in {base_name}: {e}")

# گزارش نهایی
summary_df = pd.DataFrame(all_accuracies)
summary_df.to_excel("final_batch_accuracy_with_debug.xlsx", index=False)
print("\nفرآیند تمام شد. برای بررسی دلایل افت دقت، پوشه 'debug_mismatches' رو چک کن.")

--- شروع مقایسه و عیب‌یابی برای 15 فایل ---

⚠️ Q_Cagliari_1933_91: Accuracy 94.79% (Saved 19 errors to debug folder)
⚠️ Q_Venezia_1935_186: Accuracy 92.33% (Saved 28 errors to debug folder)
⚠️ Q_Cagliari_1924_66: Accuracy 91.80% (Saved 30 errors to debug folder)
⚠️ Q_Catanzaro_1940_54: Accuracy 80.33% (Saved 72 errors to debug folder)
✅ Q_Bari_1969_26: Accuracy 100.00%
✅ Q_Pescara_1926_65: Accuracy 98.36%
⚠️ Q_Pescara_1947_73: Accuracy 96.16% (Saved 14 errors to debug folder)
⚠️ Q_Genova_1936_86: Accuracy 81.15% (Saved 69 errors to debug folder)
⚠️ Q_Venezia_1926_138: Accuracy 96.44% (Saved 13 errors to debug folder)
⚠️ Q_Parma_1942_75: Accuracy 92.05% (Saved 29 errors to debug folder)
⚠️ Q_Catanzaro_1937_36: Accuracy 83.29% (Saved 61 errors to debug folder)
⚠️ Q_Roma_1927_79: Accuracy 92.60% (Saved 27 errors to debug folder)
✅ Q_Roma_1980_30: Accuracy 98.63%
⚠️ Q_Pisa_1935_66: Accuracy 85.75% (Saved 52 errors to debug folder)
⚠️ Q_Roma_1928_73: Accuracy 95.36% (Saved 17 errors to deb